# Excess Inflation - Notebook 1: Data Acquisition

Pull ALFRED vintages, build an information-state panel, and fetch market-return proxies.


## 1. Setup & Imports


In [ ]:
# Cell 1: Drive mount (top of every notebook)
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/macro_strategies/'
import os
os.makedirs(BASE_PATH + 'data/raw', exist_ok=True)
os.makedirs(BASE_PATH + 'data/processed', exist_ok=True)


In [ ]:
# Cell 2: Standard installs
!pip install -q fredapi vectorbt yfinance pandas-ta matplotlib seaborn pyarrow


In [ ]:
# Cell 3: API keys (store in Colab Secrets, not hardcoded)
from google.colab import userdata
FRED_API_KEY = userdata.get('FRED_API_KEY')


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from src.alfred_client import (
    get_alfred_vintages, build_information_state,
    cache_vintages, load_cached_vintages,
)


## 2. ALFRED Configuration

Set API key and vintage date logic.


In [ ]:
SERIES_IDS = ['CPIAUCSL', 'CPILFESL', 'PCEPILFE', 'FEDFUNDS']
START = '1990-01-01'


## 3. Series Pull

Pull every required FRED series in ALFRED vintage format.


In [ ]:
vintages = {
    sid: get_alfred_vintages(sid, FRED_API_KEY, start=START)
    for sid in SERIES_IDS
}


## 4. Vintage Alignment

Collapse each vintage matrix into a point-in-time series and join.


In [ ]:
pit = {
    sid: build_information_state(v)['value']
    for sid, v in vintages.items()
}
panel = pd.DataFrame(pit)
panel.tail()


## 5. Market Return Data

yfinance ETF proxies (SPY, TLT, UUP, DJP, ...).


In [ ]:
tickers = ['SPY', 'TLT', 'UUP']
prices = yf.download(tickers, start=START, auto_adjust=True)['Close']
returns = prices.pct_change().dropna()
returns.tail()


## 6. Data Quality Checks

Vintage coverage heatmap and revision-magnitude charts.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

primary_sid = SERIES_IDS[0]
vdf = vintages[primary_sid]
coverage = (~vdf.isna()).astype(int)
step = max(1, coverage.shape[1] // 30)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(coverage.iloc[:, ::step].T, ax=ax, cbar=False, cmap='Blues',
            xticklabels=24, yticklabels=False)
ax.set_title(f'{primary_sid} — Vintage Coverage (blue = data available)')
plt.tight_layout()
plt.show()

rev_mag = {}
for sid in SERIES_IDS:
    v = vintages[sid]
    if v.shape[1] > 1:
        first = v.ffill(axis=1).iloc[:, 0]
        last  = v.ffill(axis=1).iloc[:, -1]
        rev_mag[sid] = (last - first).abs().mean()
if rev_mag:
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    pd.Series(rev_mag).plot(kind='bar', ax=ax2, color='steelblue')
    ax2.set_title('Mean Absolute Revision by Series')
    ax2.set_ylabel('Mean Abs Revision')
    plt.tight_layout()
    plt.show()

## 7. Export


In [ ]:
panel.to_parquet(BASE_PATH + 'data/processed/aligned_panel.parquet')
returns.to_parquet(BASE_PATH + 'data/processed/market_returns.parquet')


## Limitations

- ALFRED coverage may be incomplete pre-1995.
- yfinance ETFs are proxies, not true futures.
